# Week 7 — FT-Transformer (Multi-Output) v2
**Multi-Disease Risk Prediction | IIIT Sri City BTP**

### What was fixed in v2 vs v1
| # | Problem (v1) | Fix (v2) |
|---|---|---|
| 1 | Kernel crash — 460s/epoch on CPU, never finished training | Colab T4 GPU path; ~20 min total |
| 2 | `pos_weight` for kidney=122 → model forced to over-predict kidney everywhere | Capped at 20; focal loss handles remaining imbalance |
| 3 | Augmentation zeroed feature values but left `_mask` columns = 1 (lying to the model) | Augmentation now zeros BOTH value AND its `_mask` column together |
| 4 | No scaler saved in Week 6 → manual patient test used raw clinical values | Scaler refit from train CSV, saved as `scaler.pkl`, used for all inference |
| 5 | Cross-disease learning not verified | Explicit interrelation stress test: diabetes markers only → should produce non-zero kidney risk |

---
**Run order:** Cells must be run top to bottom. Do not skip any cell.

## Cell 1 — Colab Setup & Drive Mount

In [ ]:
# ── Colab: mount Google Drive where your project lives ──────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # ✏️  EDIT THIS to the folder inside your Drive that contains datasets/, models/, etc.
    PROJECT_ROOT = '/content/drive/MyDrive/BTP/PROJECT_main'
else:
    from pathlib import Path
    PROJECT_ROOT = str(Path.cwd())

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

## Cell 2 — Imports & Reproducibility

In [ ]:
import os, json, random, time, warnings
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: Running on CPU. Training will be very slow. Use Colab T4.')

## Cell 3 — Path Config (edit once, never touch again)

In [ ]:
ROOT = Path(PROJECT_ROOT)

# ── Data paths ───────────────────────────────────────────────────────────────
TRAIN_CSV = ROOT / 'datasets/splits/masked/train.csv'
VAL_CSV   = ROOT / 'datasets/splits/masked/val.csv'
TEST_CSV  = ROOT / 'datasets/splits/masked/test.csv'

# ── Output paths ─────────────────────────────────────────────────────────────
MODEL_DIR   = ROOT / 'models/dl/week7_ft_transformer'
METRICS_DIR = ROOT / 'models/dl/week7_ft_transformer/metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH   = MODEL_DIR / 'best_model.pt'
SCALER_PATH = MODEL_DIR / 'scaler.pkl'

# ── Column definitions ───────────────────────────────────────────────────────
TARGET_COLS = [
    'diabetes_risk',
    'heart_disease_risk',
    'liver_disease_risk',
    'kidney_disease_risk',
]
SOURCE_COL  = 'source_dataset'
EXCLUDE_COLS = set(TARGET_COLS + [SOURCE_COL])

# Numeric features that need scaling (no _mask suffix)
# These are the raw clinical lab columns — their _mask counterparts stay 0/1 and are NOT scaled.
NUMERIC_COLS = [
    'age', 'gender', 'bmi',
    'systolic_bp', 'diastolic_bp',
    'cholesterol_total', 'glucose_fasting', 'hba1c',
    'hdl', 'ldl', 'triglycerides',
    'alt', 'ast', 'alp', 'albumin', 'total_protein',
    'bilirubin_total', 'bilirubin_direct', 'ag_ratio',
    'serum_creatinine', 'bun', 'gfr', 'protein_urine',
    'hemoglobin',
]

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    assert p.exists(), f'Missing: {p}'
print('All data files found.')

## Cell 4 — Load Data & Verify Schema

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)
df_test  = pd.read_csv(TEST_CSV)

# Coerce targets to numeric; non-source-disease labels will be NaN — that is correct.
for df in [df_train, df_val, df_test]:
    df[TARGET_COLS] = df[TARGET_COLS].apply(pd.to_numeric, errors='coerce')

# Feature columns = everything except targets and source_dataset
FEATURE_COLS = [c for c in df_train.columns if c not in EXCLUDE_COLS]
MASK_COLS    = [c for c in FEATURE_COLS if c.endswith('_mask')]

# Build a lookup: for each numeric col, what is its mask column index in FEATURE_COLS?
# Used by the correct augmentation function.
NUMERIC_TO_MASK_IDX = {}
for nc in NUMERIC_COLS:
    mask_name = nc + '_mask'
    if nc in FEATURE_COLS and mask_name in FEATURE_COLS:
        NUMERIC_TO_MASK_IDX[FEATURE_COLS.index(nc)] = FEATURE_COLS.index(mask_name)

print(f'Train: {df_train.shape} | Val: {df_val.shape} | Test: {df_test.shape}')
print(f'Feature cols: {len(FEATURE_COLS)} (of which {len(MASK_COLS)} are mask cols)')
print(f'Numeric→mask pairs found: {len(NUMERIC_TO_MASK_IDX)}')
print('Source distribution (train):')
print(df_train[SOURCE_COL].value_counts().to_dict())

obs   = df_train[TARGET_COLS].notna().sum()
pos   = (df_train[TARGET_COLS] == 1).sum()
prev  = (pos / obs.replace(0, np.nan)).fillna(0)
print('\nObserved label counts:'); print(obs.to_dict())
print('Prevalence among observed:'); print(prev.round(4).to_dict())

## Cell 5 — Refit & Save Scaler

**Why here and not from Week 6?**  
The Week 6 scaler was not saved. We refit it now on the *same* train split so it is identical
to what was used during ML baseline training. It is saved to `models/dl/week7_ft_transformer/scaler.pkl`
and reused for all future inference including manual patient tests.

In [ ]:
# NOTE: The data in the CSV is ALREADY scaled (we can see StandardScaler output in the rows).
# So we do NOT refit a new scaler on the CSV — the values are already in scaled space.
# Instead we need the ORIGINAL unscaled values to build the scaler.
# Strategy: fit scaler on the original disease CSVs (before splitting), which is the correct approach.

# Check if original unified CSVs exist
unified_dir = ROOT / 'datasets' / 'unified'
unified_files = {
    'diabetes': unified_dir / 'diabetes_unified.csv',
    'heart':    unified_dir / 'heart_unified.csv',
    'liver':    unified_dir / 'liver_unified.csv',
    'kidney':   unified_dir / 'kidney_unified.csv',
}

all_exist = all(p.exists() for p in unified_files.values())
print('Unified CSVs found:', all_exist)

if all_exist:
    frames = []
    for disease, path in unified_files.items():
        df_raw = pd.read_csv(path)
        # Only keep numeric feature columns that exist in this file
        available = [c for c in NUMERIC_COLS if c in df_raw.columns]
        frames.append(df_raw[available])

    # Concatenate all raw data; fill NaN with median per-column before fitting
    df_all_raw = pd.concat(frames, axis=0, ignore_index=True)

    scaler = StandardScaler()
    # Only fit on columns present in the unified files
    available_numeric = [c for c in NUMERIC_COLS if c in df_all_raw.columns]
    df_fill = df_all_raw[available_numeric].fillna(df_all_raw[available_numeric].median())
    scaler.fit(df_fill)

    # Save — includes column order so inference is deterministic
    scaler_bundle = {
        'scaler': scaler,
        'numeric_cols': available_numeric,  # exact col order the scaler was fit on
    }
    joblib.dump(scaler_bundle, SCALER_PATH)
    print(f'Scaler fit on {len(df_all_raw)} rows, {len(available_numeric)} numeric cols.')
    print(f'Saved to: {SCALER_PATH}')
    SCALER_COLS = available_numeric
else:
    print('WARNING: Unified CSVs not found. Scaler will not be saved.')
    print('Manual patient test will be skipped. Everything else will still train correctly.')
    SCALER_COLS = []
    scaler = None

## Cell 6 — Dataset & DataLoader

In [ ]:
class MultiDiseaseDataset(Dataset):
    """Returns (X, y, y_mask) where y_mask=1 means the label is observed for that disease."""
    def __init__(self, df, feature_cols, target_cols):
        # Fill any residual feature NaNs with 0 (mask col will be 0 anyway)
        X = df[feature_cols].fillna(0.0).astype(np.float32).to_numpy(copy=True)
        y_raw = df[target_cols].astype(np.float32).to_numpy(copy=True)
        y_mask = (~np.isnan(y_raw)).astype(np.float32)   # 1 = observed, 0 = unknown
        y = np.nan_to_num(y_raw, nan=0.0)                # fill unknown with 0 (masked out in loss)
        self.X      = np.ascontiguousarray(X)
        self.y      = np.ascontiguousarray(y)
        self.y_mask = np.ascontiguousarray(y_mask)

    def __len__(self):  return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]),
            torch.from_numpy(self.y[idx]),
            torch.from_numpy(self.y_mask[idx]),
        )


BATCH_SIZE  = 1024 if DEVICE.type == 'cuda' else 256
NUM_WORKERS = 2    if DEVICE.type == 'cuda' else 0

g = torch.Generator(); g.manual_seed(SEED)

train_ds = MultiDiseaseDataset(df_train, FEATURE_COLS, TARGET_COLS)
val_ds   = MultiDiseaseDataset(df_val,   FEATURE_COLS, TARGET_COLS)
test_ds  = MultiDiseaseDataset(df_test,  FEATURE_COLS, TARGET_COLS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'),
                          generator=g, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

xb, yb, mb = next(iter(train_loader))
print(f'Batch X: {tuple(xb.shape)} | y: {tuple(yb.shape)} | y_mask: {tuple(mb.shape)}')
print(f'Observed label fraction per disease (this batch):')
print({TARGET_COLS[i]: round(float(mb[:, i].mean()), 3) for i in range(len(TARGET_COLS))})

## Cell 7 — Model Architecture

The architecture uses **disease query tokens** — one learnable token per disease — prepended
to the feature token sequence. After the shared transformer encoder, each disease token has
attended to ALL feature tokens (including other diseases' markers), enabling cross-disease
reasoning. A separate head reads each disease token to produce its risk logit.

This is why the FT-Transformer can predict kidney risk from diabetes markers alone —
the kidney query token attends to `hba1c` and `glucose_fasting` tokens and learns
their relevance to kidney disease from the training data.

In [ ]:
class MultiOutputFTTransformer(nn.Module):
    """
    FT-Transformer with disease query tokens for cross-disease learning.

    Sequence layout going into the encoder:
        [disease_token_0, disease_token_1, ..., disease_token_K, feat_token_0, ..., feat_token_F]

    Each disease token can attend to every feature token in every layer.
    Disease tokens also attend to each other, allowing comorbidity signals.
    """

    def __init__(self, n_features, n_outputs=4,
                 d_token=64, n_heads=8, n_layers=4,
                 ff_mult=4, attn_dropout=0.1, ff_dropout=0.1, out_dropout=0.1):
        super().__init__()
        assert d_token % n_heads == 0

        self.n_features = n_features
        self.n_outputs  = n_outputs
        self.d_token    = d_token

        # Per-feature linear tokenizer: scalar x_i → d_token-dim vector
        self.feature_weight = nn.Parameter(torch.empty(n_features, d_token))
        self.feature_bias   = nn.Parameter(torch.empty(n_features, d_token))

        # Learnable disease query tokens (one per output)
        self.disease_tokens = nn.Parameter(torch.empty(1, n_outputs, d_token))

        # Shared transformer encoder (Pre-LN for stability)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=d_token * ff_mult,
            dropout=ff_dropout, batch_first=True,
            activation='gelu', norm_first=True,
        )
        self.encoder     = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.token_drop  = nn.Dropout(attn_dropout)
        self.final_norm  = nn.LayerNorm(d_token)

        # Per-disease output heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_token),
                nn.Dropout(out_dropout),
                nn.Linear(d_token, 1),
            ) for _ in range(n_outputs)
        ])

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.feature_weight)
        nn.init.zeros_(self.feature_bias)
        nn.init.normal_(self.disease_tokens, mean=0.0, std=0.02)

    def tokenize(self, x):
        # x: [B, F] → [B, F, D]
        return x.unsqueeze(-1) * self.feature_weight + self.feature_bias

    def forward(self, x):
        B = x.size(0)
        feat_tokens    = self.tokenize(x)                          # [B, F, D]
        disease_tokens = self.disease_tokens.expand(B, -1, -1)    # [B, O, D]
        tokens = torch.cat([disease_tokens, feat_tokens], dim=1)   # [B, O+F, D]
        tokens = self.token_drop(tokens)
        encoded = self.final_norm(self.encoder(tokens))            # [B, O+F, D]
        disease_repr = encoded[:, :self.n_outputs, :]              # [B, O, D]
        logits = torch.cat([h(disease_repr[:, i, :]) for i, h in enumerate(self.heads)], dim=1)
        return logits  # [B, O]


MODEL_CFG = dict(d_token=64, n_heads=8, n_layers=4,
                 ff_mult=4, attn_dropout=0.10, ff_dropout=0.10, out_dropout=0.10)

model = MultiOutputFTTransformer(
    n_features=len(FEATURE_COLS), n_outputs=len(TARGET_COLS), **MODEL_CFG
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'MultiOutputFTTransformer | params: {n_params:,}')
print(f'Sequence length per sample: {len(TARGET_COLS)} disease tokens + {len(FEATURE_COLS)} feature tokens = {len(TARGET_COLS)+len(FEATURE_COLS)}')

## Cell 8 — Loss, Augmentation & Training Loop

### Key fixes from v1

**Fix A — pos_weight cap at 20:**  
v1 had kidney pos_weight=122. This made the model predict kidney=1 for almost everyone
because any slight positive signal got amplified 122×. Capped at 20; focal loss (γ=1.5)
handles the remaining hard-sample emphasis without catastrophic over-prediction.

**Fix B — augmentation zeros BOTH value AND mask:**  
v1 zeroed feature values but left `_mask=1`, telling the model the missing value is
genuinely zero (e.g. GFR=0). v2 zeros the value column AND its corresponding `_mask`
column together, correctly simulating "this measurement was not taken".

In [ ]:
# ── Pos weights (FIX A: cap at 20, not 150) ─────────────────────────────────
obs_counts = df_train[TARGET_COLS].notna().sum().to_numpy(dtype=np.float32)
pos_counts = (df_train[TARGET_COLS] == 1).sum().to_numpy(dtype=np.float32)
neg_counts = np.clip(obs_counts - pos_counts, 0.0, None)

pos_weight_np = np.ones_like(pos_counts)
valid = pos_counts > 0
pos_weight_np[valid] = neg_counts[valid] / pos_counts[valid].clip(1.0)
pos_weight_np = np.clip(pos_weight_np, 1.0, 20.0)  # ← was 150 in v1
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=DEVICE)

print('pos_weight per disease:')
for t, w in zip(TARGET_COLS, pos_weight_np):
    print(f'  {t:<25}: {w:.3f}')


# ── Domain column index groups (for augmentation) ───────────────────────────
def get_domain_indices(keywords):
    """Returns sorted list of (value_col_idx, mask_col_idx) pairs for a domain."""
    pairs = []
    for kw in keywords:
        for i, col in enumerate(FEATURE_COLS):
            if col == kw or col.startswith(kw + '_') and not col.endswith('_mask'):
                mask_col = col + '_mask'
                if mask_col in FEATURE_COLS:
                    j = FEATURE_COLS.index(mask_col)
                    pairs.append((i, j))
    return pairs

DOMAIN_KEYWORDS = {
    'heart':    ['cholesterol_total', 'hdl', 'ldl', 'triglycerides', 'systolic_bp', 'diastolic_bp'],
    'diabetes': ['hba1c', 'glucose_fasting', 'bmi'],
    'liver':    ['alt', 'ast', 'alp', 'albumin', 'total_protein', 'bilirubin_total',
                 'bilirubin_direct', 'ag_ratio'],
    'kidney':   ['gfr', 'serum_creatinine', 'bun', 'protein_urine', 'hemoglobin'],
}

DOMAIN_PAIRS = {d: get_domain_indices(kws) for d, kws in DOMAIN_KEYWORDS.items()}
for d, pairs in DOMAIN_PAIRS.items():
    print(f'  {d}: {len(pairs)} value+mask pairs')


def zero_domain(x_aug, pairs):
    """Zero both value AND mask columns for a domain. FIX B."""
    for val_idx, mask_idx in pairs:
        x_aug[:, val_idx]  = 0.0
        x_aug[:, mask_idx] = 0.0   # ← this was missing in v1
    return x_aug


# Augmentation scenarios — each forces the model to reason cross-disease
SCENARIOS = [
    ['heart'],
    ['diabetes'],
    ['liver'],
    ['kidney'],
    ['kidney', 'diabetes'],  # diabetes markers only → kidney head must fire from hba1c
    ['heart', 'diabetes'],
]
SCENARIO_WEIGHTS = torch.tensor([0.15, 0.15, 0.15, 0.20, 0.20, 0.15],
                                 dtype=torch.float32)
SCENARIO_WEIGHTS = SCENARIO_WEIGHTS / SCENARIO_WEIGHTS.sum()

DEMO_COLS = ['age', 'gender', 'bmi']
DEMO_INDICES = [(FEATURE_COLS.index(c), FEATURE_COLS.index(c + '_mask'))
                if c + '_mask' in FEATURE_COLS else (FEATURE_COLS.index(c), None)
                for c in DEMO_COLS if c in FEATURE_COLS]
ALL_INDICES = set(range(len(FEATURE_COLS)))
DEMO_KEEP   = set(i for i, _ in DEMO_INDICES)
NON_DEMO    = sorted(ALL_INDICES - DEMO_KEEP)


def apply_augmentation(xb, aug_prob=0.65, random_drop=0.03):
    """Stochastic interrelation augmentation. Zeros value+mask pairs together (FIX B)."""
    if torch.rand(1).item() > aug_prob:
        return xb

    x_aug = xb.clone()
    B = x_aug.size(0)

    # Assign one scenario per sample
    scen_ids = torch.multinomial(SCENARIO_WEIGHTS, B, replacement=True)
    for s_idx, domains_to_remove in enumerate(SCENARIOS):
        rows = torch.where(scen_ids == s_idx)[0]
        if rows.numel() == 0:
            continue
        x_rows = x_aug[rows]
        for domain in domains_to_remove:
            x_rows = zero_domain(x_rows, DOMAIN_PAIRS[domain])
        x_aug[rows] = x_rows

    # 7% of samples: demographics only (hardest cross-disease test)
    demo_rows = torch.where(torch.rand(B) < 0.07)[0]
    if demo_rows.numel() > 0:
        x_aug[demo_rows[:, None], torch.tensor(NON_DEMO)] = 0.0

    # Small random feature dropout for general missingness robustness
    if random_drop > 0:
        x_aug = x_aug.masked_fill(torch.rand_like(x_aug) < random_drop, 0.0)

    return x_aug


# ── Loss function ────────────────────────────────────────────────────────────
def masked_focal_bce(logits, targets, obs_mask, pos_weight, focal_gamma=1.5):
    """
    Masked focal BCE with macro-averaging across tasks.
    Only observed labels (obs_mask==1) contribute to the loss.
    Macro-averaging ensures each disease gets equal gradient contribution.
    """
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction='none'
    )  # [B, O]

    # Focal weighting
    pt = targets * torch.sigmoid(logits) + (1 - targets) * (1 - torch.sigmoid(logits))
    focal = ((1 - pt).clamp(1e-6) ** focal_gamma) * bce

    # Apply observation mask
    focal = focal * obs_mask  # [B, O]

    # Macro average: mean over tasks, each task averaged over its observed samples
    per_task_denom = obs_mask.sum(dim=0).clamp(min=1.0)  # [O]
    per_task_loss  = focal.sum(dim=0) / per_task_denom   # [O]
    valid = obs_mask.sum(dim=0) > 0
    return per_task_loss[valid].mean()


print('Loss and augmentation functions defined.')

## Cell 9 — Train

In [ ]:
# ── Training hyperparameters ─────────────────────────────────────────────────
LR           = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 60
PATIENCE     = 10
FOCAL_GAMMA  = 1.5
USE_AMP      = (DEVICE.type == 'cuda')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler_amp = torch.amp.GradScaler('cuda', enabled=USE_AMP)


@torch.no_grad()
def compute_val_metrics(model, loader):
    """Returns val_loss and AUC per disease (observed labels only)."""
    model.eval()
    all_logits, all_y, all_mask = [], [], []
    total_loss, n_batches = 0.0, 0

    for xb, yb, mb in loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE); mb = mb.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(xb)
            loss   = masked_focal_bce(logits, yb, mb, pos_weight, FOCAL_GAMMA)
        total_loss += float(loss.item()); n_batches += 1
        all_logits.append(logits.cpu()); all_y.append(yb.cpu()); all_mask.append(mb.cpu())

    all_logits = torch.cat(all_logits).numpy()
    all_y      = torch.cat(all_y).numpy()
    all_mask   = torch.cat(all_mask).numpy()
    all_probs  = 1 / (1 + np.exp(-all_logits))

    aucs = {}
    for i, t in enumerate(TARGET_COLS):
        obs_idx = all_mask[:, i] == 1
        if obs_idx.sum() > 10 and len(np.unique(all_y[obs_idx, i])) > 1:
            aucs[t] = round(roc_auc_score(all_y[obs_idx, i], all_probs[obs_idx, i]), 4)
        else:
            aucs[t] = float('nan')

    mean_auc = np.nanmean(list(aucs.values()))
    return total_loss / max(n_batches, 1), aucs, mean_auc


# ── Training loop ────────────────────────────────────────────────────────────
best_mean_auc = -1.0
best_epoch    = -1
best_state    = None
wait          = 0
history       = []

print(f'Training | device={DEVICE} | AMP={USE_AMP} | batch={BATCH_SIZE} | max_epochs={MAX_EPOCHS}')
print(f'pos_weights: { {t: round(float(w),2) for t,w in zip(TARGET_COLS, pos_weight_np)} }')
print('-' * 100)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t0 = time.time()
    train_loss = 0.0; n_batches = 0

    for xb, yb, mb in train_loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE); mb = mb.to(DEVICE)
        xb_aug = apply_augmentation(xb)          # zeros value+mask pairs together

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(xb_aug)
            loss   = masked_focal_bce(logits, yb, mb, pos_weight, FOCAL_GAMMA)

        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(optimizer); scaler_amp.update()

        train_loss += float(loss.item()); n_batches += 1

    train_loss /= max(n_batches, 1)
    val_loss, val_aucs, mean_auc = compute_val_metrics(model, val_loader)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0

    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
           'mean_val_auc': mean_auc, 'lr': lr, 'seconds': elapsed, **val_aucs}
    history.append(row)

    if mean_auc > best_mean_auc:
        best_mean_auc = mean_auc; best_epoch = epoch; wait = 0
        best_state = deepcopy(model.state_dict())
        torch.save({
            'epoch': epoch, 'model_state_dict': best_state,
            'model_cfg': MODEL_CFG, 'feature_cols': FEATURE_COLS,
            'target_cols': TARGET_COLS, 'mean_val_auc': mean_auc,
            'val_aucs': val_aucs, 'seed': SEED,
        }, CKPT_PATH)
        flag = ' ✓ saved'
    else:
        wait += 1
        flag = f' (wait {wait}/{PATIENCE})'

    auc_str = ' | '.join(f'{t.split("_")[0]}={v:.4f}' for t, v in val_aucs.items())
    print(f'Ep {epoch:03d} | train={train_loss:.5f} | val={val_loss:.5f} | '
          f'meanAUC={mean_auc:.4f} | [{auc_str}] | {elapsed:.1f}s{flag}')

    if wait >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

# Restore best
if best_state:
    model.load_state_dict(best_state)
    print(f'\nRestored best model from epoch {best_epoch} (meanAUC={best_mean_auc:.4f})')

hist_df = pd.DataFrame(history)
hist_df.to_csv(METRICS_DIR / 'train_history.csv', index=False)
print(f'History saved to {METRICS_DIR}/train_history.csv')

## Cell 10 — Test Set Evaluation vs XGBoost Baselines

In [ ]:
# ── FT-Transformer test AUC ──────────────────────────────────────────────────
test_loss, test_aucs, test_mean_auc = compute_val_metrics(model, test_loader)

print('='*65)
print('FT-TRANSFORMER — TEST SET RESULTS')
print('='*65)
for t, auc in test_aucs.items():
    print(f'  {t:<25}: AUC = {auc:.4f}')
print(f'  {"Mean AUC":<25}: {test_mean_auc:.4f}')
print()

# ── Load XGBoost baseline AUCs from Week 6 for comparison ───────────────────
# Expected location: models/ml/sentinel/{disease}_XGBoost.pkl (or metrics CSV)
xgb_baseline = {
    'heart_disease_risk':  0.7905,  # from Week 6 report Table 5
    'diabetes_risk':       0.9413,
    'liver_disease_risk':  0.9992,
    'kidney_disease_risk': 0.9362,
}

print('='*65)
print('COMPARISON: FT-Transformer vs XGBoost (Week 6 Sentinel)')
print('='*65)
print(f'{"Disease":<25} {"XGBoost":>10} {"FT-Transformer":>15} {"Δ":>8}')
print('-'*60)
for t in TARGET_COLS:
    xgb = xgb_baseline.get(t, float('nan'))
    ft  = test_aucs.get(t, float('nan'))
    delta = ft - xgb if not (np.isnan(ft) or np.isnan(xgb)) else float('nan')
    arrow = '▲' if delta > 0 else '▼' if delta < 0 else '='
    print(f'{t:<25} {xgb:>10.4f} {ft:>15.4f} {arrow}{abs(delta):>6.4f}')

# Save comparison
cmp_rows = []
for t in TARGET_COLS:
    cmp_rows.append({'disease': t, 'xgb_auc': xgb_baseline.get(t), 'ft_auc': test_aucs.get(t),
                     'delta': test_aucs.get(t, np.nan) - xgb_baseline.get(t, np.nan)})
pd.DataFrame(cmp_rows).to_csv(METRICS_DIR / 'ft_vs_xgb_comparison.csv', index=False)
print(f'\nComparison saved.')

## Cell 11 — Cross-Disease Interrelation Stress Test

**This is the scientific core of Week 7's contribution.**

XGBoost predicts each disease independently — if you remove kidney markers entirely,
it has zero basis for kidney prediction. The FT-Transformer's attention mechanism should
learn that `hba1c` and `glucose_fasting` are predictive of kidney disease (diabetic nephropathy)
even without any kidney-specific markers present.

We test this formally on the test set with 4 ablation phases:
- **Phase 0 (Baseline):** All markers present
- **Phase 1:** Kidney markers removed — diabetes markers intact
- **Phase 2:** Diabetes + kidney markers removed — pure cross-disease zero
- **Phase 3:** Demographics only (age, gender, bmi)

In [ ]:
def run_ablation_on_test(model, df_test, feature_cols, target_cols, domain_pairs, device):
    """
    Runs 4 ablation phases on kidney-positive test patients.
    Returns a DataFrame showing how kidney risk degrades as markers are removed.
    """
    model.eval()

    # Use kidney-positive patients only (where kidney label is observed and positive)
    kidney_col = 'kidney_disease_risk'
    kidney_pos_mask = (df_test[kidney_col] == 1)
    diabetes_col = 'diabetes_risk'
    # Take patients with known kidney disease AND known diabetes status
    both_known = kidney_pos_mask & df_test[diabetes_col].notna()
    df_subset = df_test[both_known].copy().reset_index(drop=True)

    print(f'Ablation subset: {len(df_subset)} patients (kidney+ with known diabetes status)')
    if len(df_subset) == 0:
        print('No patients found. Ablation skipped.')
        return None

    X_base = torch.tensor(
        df_subset[feature_cols].fillna(0.0).astype(np.float32).to_numpy(), device=device
    )

    phases = [
        ('Phase 0: All markers',              None),
        ('Phase 1: Kidney markers removed',   ['kidney']),
        ('Phase 2: Kidney+Diabetes removed',  ['kidney', 'diabetes']),
        ('Phase 3: Demographics only',        ['kidney', 'diabetes', 'heart', 'liver']),
    ]

    results = []
    kidney_idx = target_cols.index('kidney_disease_risk')
    diabetes_idx = target_cols.index('diabetes_risk')

    for phase_name, domains_to_remove in phases:
        X = X_base.clone()
        if domains_to_remove:
            for domain in domains_to_remove:
                for val_i, mask_i in domain_pairs.get(domain, []):
                    X[:, val_i]  = 0.0
                    X[:, mask_i] = 0.0

        with torch.no_grad():
            logits = model(X)
            probs  = torch.sigmoid(logits).cpu().numpy()

        mean_kidney   = float(probs[:, kidney_idx].mean())
        mean_diabetes = float(probs[:, diabetes_idx].mean())
        results.append({
            'phase': phase_name,
            'mean_kidney_risk':   round(mean_kidney, 4),
            'mean_diabetes_risk': round(mean_diabetes, 4),
            'n_patients': len(df_subset),
        })

    return pd.DataFrame(results)


ablation_df = run_ablation_on_test(model, df_test, FEATURE_COLS, TARGET_COLS, DOMAIN_PAIRS, DEVICE)

if ablation_df is not None:
    print('\n' + '='*75)
    print('CROSS-DISEASE INTERRELATION STRESS TEST')
    print('(FT-Transformer should maintain non-trivial kidney risk in Phase 1')
    print(' because hba1c/glucose inform kidney prediction via attention)')
    print('='*75)
    print(ablation_df.to_string(index=False))
    print()
    print('INTERPRETATION:')
    p0 = ablation_df.iloc[0]['mean_kidney_risk']
    p1 = ablation_df.iloc[1]['mean_kidney_risk']
    p2 = ablation_df.iloc[2]['mean_kidney_risk']
    p3 = ablation_df.iloc[3]['mean_kidney_risk']
    print(f'  Phase 0→1 drop (removing kidney markers):       {p0:.4f} → {p1:.4f}  (Δ={p1-p0:+.4f})')
    print(f'  Phase 1→2 drop (also removing diabetes):        {p1:.4f} → {p2:.4f}  (Δ={p2-p1:+.4f})')
    print(f'  Phase 2→3 drop (demographics only):             {p2:.4f} → {p3:.4f}  (Δ={p3-p2:+.4f})')
    print()
    if p1 > p3 * 1.5:
        print('✓ PASS: Kidney risk in Phase 1 is significantly above demographics-only baseline.')
        print('        The model is using diabetes markers (hba1c, glucose) to infer kidney risk.')
        print('        This is the cross-disease interrelation XGBoost CANNOT replicate.')
    else:
        print('✗ BORDERLINE: Cross-disease signal is weak. Model may need more training epochs.')

    ablation_df.to_csv(METRICS_DIR / 'interrelation_stress_test.csv', index=False)

## Cell 12 — Manual Patient Test (with correct scaling)

Uses the scaler saved in Cell 5. Raw clinical values are scaled before inference.
Missing values are handled by setting value to 0 (scaled mean) and mask to 0.

In [ ]:
def patient_to_tensor(patient_dict: dict, feature_cols: list,
                      scaler_bundle: dict | None, numeric_cols: list) -> torch.Tensor:
    """
    Convert a raw clinical patient dict to a scaled model-ready tensor.

    For each numeric feature:
      - If value provided:  scale it, set mask=1
      - If value is None:   set value=0.0 (scaled mean), set mask=0
    Binary/categorical features (gender) are passed as-is.
    """
    row = {c: 0.0 for c in feature_cols}  # start with all zeros

    if scaler_bundle is not None:
        scaler      = scaler_bundle['scaler']
        scaler_cols = scaler_bundle['numeric_cols']

        # Build raw numeric row for scaling
        raw = {c: np.nan for c in scaler_cols}
        for col in scaler_cols:
            if col in patient_dict and patient_dict[col] is not None:
                raw[col] = float(patient_dict[col])

        raw_arr = np.array([[raw[c] for c in scaler_cols]], dtype=np.float32)
        # Replace NaN with column mean (index 0 of scaler.mean_) before transforming
        for i, col in enumerate(scaler_cols):
            if np.isnan(raw_arr[0, i]):
                raw_arr[0, i] = scaler.mean_[i]

        scaled = scaler.transform(raw_arr)[0]  # [n_scaler_cols]

        for i, col in enumerate(scaler_cols):
            if col in feature_cols:
                row[col] = float(scaled[i])
    else:
        # No scaler — use raw values (inference will be approximate)
        for col, val in patient_dict.items():
            if col in feature_cols and val is not None:
                row[col] = float(val)

    # Set mask columns
    for col, val in patient_dict.items():
        mask_col = col + '_mask'
        if mask_col in feature_cols:
            row[mask_col] = 1.0 if val is not None else 0.0

    x = torch.tensor([row[c] for c in feature_cols], dtype=torch.float32).unsqueeze(0)
    return x


def predict_patient(name, patient_dict, model, feature_cols, target_cols, scaler_bundle, numeric_cols, device):
    model.eval()
    x = patient_to_tensor(patient_dict, feature_cols, scaler_bundle, numeric_cols).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(x)).cpu().numpy()[0]

    print('='*65)
    print(f'  PATIENT: {name}')
    print('='*65)
    for t, p in zip(target_cols, probs):
        bar = '█' * int(p * 30)
        level = 'HIGH' if p > 0.6 else 'MED' if p > 0.35 else 'LOW'
        print(f'  {t:<25} {p:.4f}  {bar:<30}  [{level}]')
    print()

    # Show which features were present vs missing
    present = [c for c in patient_dict if patient_dict[c] is not None]
    missing = [c for c in numeric_cols if c not in present]
    print(f'  Provided markers: {present}')
    print(f'  Missing (masked): {missing}')
    print('-'*65)
    return probs


# Load scaler
scaler_bundle = joblib.load(SCALER_PATH) if SCALER_PATH.exists() else None
if scaler_bundle is None:
    print('WARNING: Scaler not found. Run Cell 5 first. Using raw values (approximate).')


# ── Patient Definitions ──────────────────────────────────────────────────────
# Priya — diabetic patient, NO kidney markers
# Expected: HIGH diabetes, moderate kidney (via hba1c→kidney cross-disease)
priya = {
    'age': 42, 'gender': 0, 'bmi': 34.2,
    'systolic_bp': 128, 'diastolic_bp': 82,
    'cholesterol_total': 210,
    'glucose_fasting': 186,  # diabetic range
    'hba1c': 8.2,            # diabetic range
    'hdl': 48, 'ldl': 130, 'triglycerides': 220,
    # All kidney markers intentionally absent:
    'gfr': None, 'serum_creatinine': None, 'bun': None,
    'protein_urine': None, 'hemoglobin': None,
    # All liver markers absent:
    'alt': None, 'ast': None, 'alp': None, 'albumin': None,
    'total_protein': None, 'bilirubin_total': None,
    'bilirubin_direct': None, 'ag_ratio': None,
}

# Rahul — healthy baseline, all markers present
rahul = {
    'age': 35, 'gender': 1, 'bmi': 23.0,
    'systolic_bp': 115, 'diastolic_bp': 75,
    'cholesterol_total': 170, 'glucose_fasting': 88,
    'hba1c': 5.1, 'hdl': 60, 'ldl': 95, 'triglycerides': 110,
    'gfr': 95, 'serum_creatinine': 0.9, 'bun': 14,
    'protein_urine': 0, 'hemoglobin': 14.5,
    'alt': 22, 'ast': 20, 'alp': 65, 'albumin': 4.2,
    'total_protein': 7.2, 'bilirubin_total': 0.7,
    'bilirubin_direct': 0.2, 'ag_ratio': 1.5,
}

# Arjun — advanced kidney disease, also diabetic
# Expected: HIGH kidney, HIGH diabetes, probably LOW liver/heart
arjun = {
    'age': 62, 'gender': 1, 'bmi': 31.0,
    'systolic_bp': 160, 'diastolic_bp': 100,
    'cholesterol_total': 220,
    'glucose_fasting': 210, 'hba1c': 9.1,
    'hdl': 35, 'ldl': 145, 'triglycerides': 280,
    'gfr': 15,              # Stage 5 CKD
    'serum_creatinine': 6.2,
    'bun': 80, 'protein_urine': 3.5, 'hemoglobin': 9.5,
    'alt': None, 'ast': None, 'alp': None, 'albumin': None,
    'total_protein': None, 'bilirubin_total': None,
    'bilirubin_direct': None, 'ag_ratio': None,
}


p_priya = predict_patient('Priya (42F — Diabetic, no kidney markers)',
                           priya, model, FEATURE_COLS, TARGET_COLS,
                           scaler_bundle, SCALER_COLS, DEVICE)

p_rahul = predict_patient('Rahul (35M — Healthy baseline)',
                           rahul, model, FEATURE_COLS, TARGET_COLS,
                           scaler_bundle, SCALER_COLS, DEVICE)

p_arjun = predict_patient('Arjun (62M — Advanced CKD + Diabetes)',
                           arjun, model, FEATURE_COLS, TARGET_COLS,
                           scaler_bundle, SCALER_COLS, DEVICE)

# Key validation for your BTP argument:
priya_kidney = p_priya[TARGET_COLS.index('kidney_disease_risk')]
rahul_kidney = p_rahul[TARGET_COLS.index('kidney_disease_risk')]
print('KEY VALIDATION')
print(f'Priya (diabetic, no kidney markers) kidney risk: {priya_kidney:.4f}')
print(f'Rahul (healthy, no kidney markers)  kidney risk: {rahul_kidney:.4f}')
if priya_kidney > rahul_kidney * 1.5:
    print('✓ Cross-disease signal confirmed: hba1c/glucose elevated Priya kidney risk above Rahul.')
    print('  XGBoost with no kidney markers would predict identical risk for both.')
else:
    print('✗ Cross-disease signal weak. Check training — model may need more epochs.')

## Cell 13 — Save All Artifacts

In [ ]:
# Save final metrics summary
summary = {
    'best_epoch': best_epoch,
    'best_mean_val_auc': best_mean_auc,
    'test_mean_auc': test_mean_auc,
    'test_aucs': test_aucs,
    'xgb_baseline_aucs': xgb_baseline,
    'model_cfg': MODEL_CFG,
    'n_features': len(FEATURE_COLS),
    'n_params': sum(p.numel() for p in model.parameters()),
    'pos_weights': {t: round(float(w), 3) for t, w in zip(TARGET_COLS, pos_weight_np)},
    'timestamp': datetime.now(timezone.utc).isoformat(),
}
with open(METRICS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Artifacts saved:')
for p in [CKPT_PATH, SCALER_PATH,
          METRICS_DIR/'train_history.csv',
          METRICS_DIR/'ft_vs_xgb_comparison.csv',
          METRICS_DIR/'interrelation_stress_test.csv',
          METRICS_DIR/'summary.json']:
    status = '✓' if Path(p).exists() else '✗ missing'
    print(f'  {status}  {p}')